In [ ]:
%%writefile fit_utils/build_user_nutrition_profile.py
from __future__ import annotations

from dataclasses import dataclass, asdict
from datetime import date, datetime
from typing import Optional, Dict, Any, Literal


GoalType = Literal[
    "weight_loss",
    "muscle_gain",
    "health",
    "competition",
    "performance",
    "aggressive_cut",
]

SexType = Literal["male", "female"]
ActivityLevelType = Literal[
    "sedentary",     # little/no exercise
    "light",         # 1-3 times/week
    "moderate",      # 3-5 times/week
    "very_active",   # 6-7 times/week
    "athlete",       # 2x/day / intense
]


@dataclass
class UserProfileInput:
    # system / ask fields
    line_user_id: str
    email: Optional[str]
    first_name: Optional[str]
    last_name: Optional[str]
    nickname: Optional[str]

    weight_kg: float
    sex: SexType
    height_cm: int
    birth_date: date

    body_fat_percent: Optional[float] = None
    know_your_bmr: Optional[float] = None  # user input BMR override

    meals_per_day: int = 3
    activity_level: ActivityLevelType = "moderate"
    goal: GoalType = "health"

    concern: Optional[str] = None
    favorite: Optional[str] = None

    # optional user override
    tdee_override: Optional[float] = None


def calculate_age(birth_date: date, today: Optional[date] = None) -> int:
    today = today or date.today()
    age = today.year - birth_date.year
    if (today.month, today.day) < (birth_date.month, birth_date.day):
        age -= 1
    return age


def calculate_bmr_mifflin_st_jeor(
    sex: SexType,
    weight_kg: float,
    height_cm: int,
    age: int,
) -> float:
    """
    Male   : BMR = 10W + 6.25H - 5A + 5
    Female : BMR = 10W + 6.25H - 5A - 161
    """
    base = (10 * weight_kg) + (6.25 * height_cm) - (5 * age)
    if sex == "male":
        return base + 5
    return base - 161


def calculate_bmr_katch_mcardle(weight_kg: float, body_fat_percent: float) -> float:
    """
    BMR = 370 + (21.6 × LBM)
    LBM = weight × (1 - body_fat%)
    """
    lbm = weight_kg * (1 - body_fat_percent / 100)
    return 370 + (21.6 * lbm)


def activity_multiplier(activity_level: ActivityLevelType) -> float:
    mapping = {
        "sedentary": 1.20,
        "light": 1.375,
        "moderate": 1.55,
        "very_active": 1.725,
        "athlete": 1.90,
    }
    return mapping[activity_level]


def recommended_goal_calorie_adjustment(goal: GoalType) -> int:
    """
    Simple default calorie adjustment from TDEE.
    You can tune this later.
    """
    mapping = {
        "weight_loss": -500,
        "aggressive_cut": -700,
        "muscle_gain": 250,
        "health": 0,
        "performance": 150,
        "competition": 200,
    }
    return mapping[goal]


def protein_range_g_per_kg(goal: GoalType, activity_level: ActivityLevelType) -> tuple[float, float]:
    """
    Based on your table + practical fallback.
    """
    if goal == "aggressive_cut":
        return (2.2, 2.6)
    if goal == "competition":
        return (2.2, 3.0)
    if goal == "weight_loss":
        return (2.0, 2.4)
    if goal == "muscle_gain":
        return (1.6, 2.2)
    if goal == "performance":
        return (1.6, 2.2)
    # health / default by activity
    if activity_level == "sedentary":
        return (0.8, 1.0)
    if activity_level == "light":
        return (1.2, 1.6)
    return (1.2, 1.8)


def fat_range_g_per_kg(goal: GoalType, activity_level: ActivityLevelType) -> tuple[float, float]:
    """
    Based on your table + practical fallback.
    """
    if goal == "aggressive_cut":
        return (0.5, 0.6)
    if goal == "weight_loss":
        return (0.6, 0.8)
    if goal == "muscle_gain":
        return (0.6, 0.8)
    if goal == "performance":
        return (0.6, 0.9)
    if goal == "competition":
        return (0.6, 0.8)
    # health / default
    if activity_level == "sedentary":
        return (0.8, 1.0)
    return (0.6, 0.8)


def choose_midpoint(low: float, high: float) -> float:
    return (low + high) / 2


def estimate_fiber_g_per_day(target_calories: float, sex: SexType) -> int:
    """
    Practical rule:
    - 14 g per 1000 kcal
    with a reasonable minimum floor:
    - male >= 30 g
    - female >= 25 g
    """
    base = round((target_calories / 1000) * 14)
    floor = 30 if sex == "male" else 25
    return max(base, floor)


def estimate_sodium_mg_per_day(goal: GoalType, activity_level: ActivityLevelType) -> int:
    """
    Generic default, not personalized for sweat rate/medical condition.
    Usually <= 2000-2300 mg/day for general population.
    """
    if goal in {"competition", "performance"} or activity_level in {"very_active", "athlete"}:
        return 2300
    return 2000


def validate_input(profile: UserProfileInput) -> None:
    if profile.weight_kg <= 0:
        raise ValueError("weight_kg must be > 0")
    if profile.height_cm <= 0:
        raise ValueError("height_cm must be > 0")
    if profile.meals_per_day <= 0:
        raise ValueError("meals_per_day must be > 0")
    if profile.body_fat_percent is not None:
        if not (3 <= profile.body_fat_percent <= 70):
            raise ValueError("body_fat_percent should be between 3 and 70")
    if profile.know_your_bmr is not None and profile.know_your_bmr <= 0:
        raise ValueError("know_your_bmr must be > 0")
    if profile.tdee_override is not None and profile.tdee_override <= 0:
        raise ValueError("tdee_override must be > 0")


def build_user_nutrition_profile(profile: UserProfileInput) -> Dict[str, Any]:
    validate_input(profile)

    age = calculate_age(profile.birth_date)

    # 1) BMR
    if profile.know_your_bmr is not None:
        bmr = float(profile.know_your_bmr)
        bmr_method = "user_override"
    else:
        if profile.body_fat_percent is not None:
            bmr = calculate_bmr_katch_mcardle(profile.weight_kg, profile.body_fat_percent)
            bmr_method = "katch_mcardle"
        else:
            bmr = calculate_bmr_mifflin_st_jeor(
                sex=profile.sex,
                weight_kg=profile.weight_kg,
                height_cm=profile.height_cm,
                age=age,
            )
            bmr_method = "mifflin_st_jeor"

    # 2) TDEE
    if profile.tdee_override is not None:
        tdee = float(profile.tdee_override)
        tdee_method = "user_override"
    else:
        tdee = bmr * activity_multiplier(profile.activity_level)
        tdee_method = "bmr_x_activity_multiplier"

    # 3) Target calories
    calorie_adjustment = recommended_goal_calorie_adjustment(profile.goal)
    target_calories = tdee + calorie_adjustment

    # safety floor (basic guardrail only)
    # You may tune this later.
    if profile.sex == "male":
        target_calories = max(target_calories, 1200)
    else:
        target_calories = max(target_calories, 1000)

    # 4) Protein/Fat recommendation by goal
    p_low, p_high = protein_range_g_per_kg(profile.goal, profile.activity_level)
    f_low, f_high = fat_range_g_per_kg(profile.goal, profile.activity_level)

    protein_g_per_kg = choose_midpoint(p_low, p_high)
    fat_g_per_kg = choose_midpoint(f_low, f_high)

    protein_g_day = round(profile.weight_kg * protein_g_per_kg)
    fat_g_day = round(profile.weight_kg * fat_g_per_kg)

    protein_kcal_day = protein_g_day * 4
    fat_kcal_day = fat_g_day * 9

    # 5) Carb = remaining calories
    remaining_kcal = target_calories - protein_kcal_day - fat_kcal_day
    carb_g_day = max(round(remaining_kcal / 4), 0)
    carb_kcal_day = carb_g_day * 4

    # 6) Fiber / Sodium
    fiber_g_day = estimate_fiber_g_per_day(target_calories, profile.sex)
    sodium_mg_day = estimate_sodium_mg_per_day(profile.goal, profile.activity_level)

    # 7) Per meal
    meals = profile.meals_per_day
    calories_per_meal = round(target_calories / meals)
    protein_g_per_meal = round(protein_g_day / meals)
    carb_g_per_meal = round(carb_g_day / meals)
    fat_g_per_meal = round(fat_g_day / meals)
    fiber_g_per_meal = round(fiber_g_day / meals)
    sodium_mg_per_meal = round(sodium_mg_day / meals)

    protein_kcal_per_meal = protein_g_per_meal * 4
    carb_kcal_per_meal = carb_g_per_meal * 4
    fat_kcal_per_meal = fat_g_per_meal * 9

    return {
        # original profile
        "line_user_id": profile.line_user_id,
        "email": profile.email,
        "first_name": profile.first_name,
        "last_name": profile.last_name,
        "nickname": profile.nickname,
        "weight_kg": profile.weight_kg,
        "sex": profile.sex,
        "height_cm": profile.height_cm,
        "birth_date": profile.birth_date.isoformat(),
        "body_fat_percent": profile.body_fat_percent,
        "meals_per_day": profile.meals_per_day,
        "activity_level": profile.activity_level,
        "goal": profile.goal,
        "concern": profile.concern,
        "favorite": profile.favorite,

        # derived
        "age": age,
        "bmr": round(bmr),
        "bmr_method": bmr_method,
        "tdee": round(tdee),
        "tdee_method": tdee_method,
        "goal_calorie_adjustment": calorie_adjustment,
        "daily_calories_intake": round(target_calories),

        # ranges used
        "protein_g_per_kg_range": [p_low, p_high],
        "fat_g_per_kg_range": [f_low, f_high],
        "protein_g_per_kg_selected": protein_g_per_kg,
        "fat_g_per_kg_selected": fat_g_per_kg,

        # per day macro in grams
        "protein_g_per_day": protein_g_day,
        "carb_g_per_day": carb_g_day,
        "fat_g_per_day": fat_g_day,
        "fiber_g_per_day": fiber_g_day,
        "sodium_mg_per_day": sodium_mg_day,

        # per day macro in calories
        "protein_kcal_per_day": protein_kcal_day,
        "carb_kcal_per_day": carb_kcal_day,
        "fat_kcal_per_day": fat_kcal_day,

        # per meal in grams/mg
        "protein_g_per_meal": protein_g_per_meal,
        "carb_g_per_meal": carb_g_per_meal,
        "fat_g_per_meal": fat_g_per_meal,
        "fiber_g_per_meal": fiber_g_per_meal,
        "sodium_mg_per_meal": sodium_mg_per_meal,

        # per meal in kcal
        "calories_per_meal": calories_per_meal,
        "protein_kcal_per_meal": protein_kcal_per_meal,
        "carb_kcal_per_meal": carb_kcal_per_meal,
        "fat_kcal_per_meal": fat_kcal_per_meal,
    }

In [ ]:
from fit_utils.build_user_nutrition_profile import build_user_nutrition_profile

# TEST CASE

## ✅ 1. Basic (ไม่มี body fat → ใช้ Mifflin)

In [ ]:
user = UserProfileInput(
    line_user_id="test1",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=95,
    sex="male",
    height_cm=180,
    birth_date=date(2000, 1, 1),
    meals_per_day=3,
    activity_level="moderate",
    goal="weight_loss"
)

result = build_user_nutrition_profile(user)
result

{'line_user_id': 'test1',
 'email': None,
 'first_name': 'A',
 'last_name': 'B',
 'nickname': 'C',
 'weight_kg': 95,
 'sex': 'male',
 'height_cm': 180,
 'birth_date': '2000-01-01',
 'body_fat_percent': None,
 'meals_per_day': 3,
 'activity_level': 'moderate',
 'goal': 'weight_loss',
 'concern': None,
 'favorite': None,
 'age': 26,
 'bmr': 1950,
 'bmr_method': 'mifflin_st_jeor',
 'tdee': 3022,
 'tdee_method': 'bmr_x_activity_multiplier',
 'goal_calorie_adjustment': -500,
 'daily_calories_intake': 2522,
 'protein_g_per_kg_range': [2.0, 2.4],
 'fat_g_per_kg_range': [0.6, 0.8],
 'protein_g_per_kg_selected': 2.2,
 'fat_g_per_kg_selected': 0.7,
 'protein_g_per_day': 209,
 'carb_g_per_day': 273,
 'fat_g_per_day': 66,
 'fiber_g_per_day': 35,
 'sodium_mg_per_day': 2000,
 'protein_kcal_per_day': 836,
 'carb_kcal_per_day': 1092,
 'fat_kcal_per_day': 594,
 'protein_g_per_meal': 70,
 'carb_g_per_meal': 91,
 'fat_g_per_meal': 22,
 'fiber_g_per_meal': 12,
 'sodium_mg_per_meal': 667,
 'calories_per_me

## ✅ 2. มี body fat → ใช้ Katch-McArdle

In [ ]:
user = UserProfileInput(
    line_user_id="test2",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=95,
    sex="male",
    height_cm=180,
    birth_date=date(2000, 1, 1),
    body_fat_percent=25,
    meals_per_day=3,
    activity_level="moderate",
    goal="weight_loss"
)
result = build_user_nutrition_profile(user)
result

{'line_user_id': 'test2',
 'email': None,
 'first_name': 'A',
 'last_name': 'B',
 'nickname': 'C',
 'weight_kg': 95,
 'sex': 'male',
 'height_cm': 180,
 'birth_date': '2000-01-01',
 'body_fat_percent': 25,
 'meals_per_day': 3,
 'activity_level': 'moderate',
 'goal': 'weight_loss',
 'concern': None,
 'favorite': None,
 'age': 26,
 'bmr': 1909,
 'bmr_method': 'katch_mcardle',
 'tdee': 2959,
 'tdee_method': 'bmr_x_activity_multiplier',
 'goal_calorie_adjustment': -500,
 'daily_calories_intake': 2459,
 'protein_g_per_kg_range': [2.0, 2.4],
 'fat_g_per_kg_range': [0.6, 0.8],
 'protein_g_per_kg_selected': 2.2,
 'fat_g_per_kg_selected': 0.7,
 'protein_g_per_day': 209,
 'carb_g_per_day': 257,
 'fat_g_per_day': 66,
 'fiber_g_per_day': 34,
 'sodium_mg_per_day': 2000,
 'protein_kcal_per_day': 836,
 'carb_kcal_per_day': 1028,
 'fat_kcal_per_day': 594,
 'protein_g_per_meal': 70,
 'carb_g_per_meal': 86,
 'fat_g_per_meal': 22,
 'fiber_g_per_meal': 11,
 'sodium_mg_per_meal': 667,
 'calories_per_meal':

In [ ]:
user = UserProfileInput(
    line_user_id="test3",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=95,
    sex="male",
    height_cm=180,
    birth_date=date(2000, 1, 1),
    know_your_bmr=1900,
    meals_per_day=3,
    activity_level="light",
    goal="health"
)
result = build_user_nutrition_profile(user)
result

{'line_user_id': 'test3',
 'email': None,
 'first_name': 'A',
 'last_name': 'B',
 'nickname': 'C',
 'weight_kg': 95,
 'sex': 'male',
 'height_cm': 180,
 'birth_date': '2000-01-01',
 'body_fat_percent': None,
 'meals_per_day': 3,
 'activity_level': 'light',
 'goal': 'health',
 'concern': None,
 'favorite': None,
 'age': 26,
 'bmr': 1900,
 'bmr_method': 'user_override',
 'tdee': 2612,
 'tdee_method': 'bmr_x_activity_multiplier',
 'goal_calorie_adjustment': 0,
 'daily_calories_intake': 2612,
 'protein_g_per_kg_range': [1.2, 1.6],
 'fat_g_per_kg_range': [0.6, 0.8],
 'protein_g_per_kg_selected': 1.4,
 'fat_g_per_kg_selected': 0.7,
 'protein_g_per_day': 133,
 'carb_g_per_day': 372,
 'fat_g_per_day': 66,
 'fiber_g_per_day': 37,
 'sodium_mg_per_day': 2000,
 'protein_kcal_per_day': 532,
 'carb_kcal_per_day': 1488,
 'fat_kcal_per_day': 594,
 'protein_g_per_meal': 44,
 'carb_g_per_meal': 124,
 'fat_g_per_meal': 22,
 'fiber_g_per_meal': 12,
 'sodium_mg_per_meal': 667,
 'calories_per_meal': 871,
 '

## ✅ 3. User ใส่ BMR เอง

In [ ]:
user = UserProfileInput(
    line_user_id="test3",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=95,
    sex="male",
    height_cm=180,
    birth_date=date(2000, 1, 1),
    know_your_bmr=1900,
    meals_per_day=3,
    activity_level="light",
    goal="health"
)
result = build_user_nutrition_profile(user)
result

{'line_user_id': 'test3',
 'email': None,
 'first_name': 'A',
 'last_name': 'B',
 'nickname': 'C',
 'weight_kg': 95,
 'sex': 'male',
 'height_cm': 180,
 'birth_date': '2000-01-01',
 'body_fat_percent': None,
 'meals_per_day': 3,
 'activity_level': 'light',
 'goal': 'health',
 'concern': None,
 'favorite': None,
 'age': 26,
 'bmr': 1900,
 'bmr_method': 'user_override',
 'tdee': 2612,
 'tdee_method': 'bmr_x_activity_multiplier',
 'goal_calorie_adjustment': 0,
 'daily_calories_intake': 2612,
 'protein_g_per_kg_range': [1.2, 1.6],
 'fat_g_per_kg_range': [0.6, 0.8],
 'protein_g_per_kg_selected': 1.4,
 'fat_g_per_kg_selected': 0.7,
 'protein_g_per_day': 133,
 'carb_g_per_day': 372,
 'fat_g_per_day': 66,
 'fiber_g_per_day': 37,
 'sodium_mg_per_day': 2000,
 'protein_kcal_per_day': 532,
 'carb_kcal_per_day': 1488,
 'fat_kcal_per_day': 594,
 'protein_g_per_meal': 44,
 'carb_g_per_meal': 124,
 'fat_g_per_meal': 22,
 'fiber_g_per_meal': 12,
 'sodium_mg_per_meal': 667,
 'calories_per_meal': 871,
 '

## ✅ 4. User ใส่ TDEE เอง (override สูงสุด)

In [ ]:
user = UserProfileInput(
    line_user_id="test4",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=95,
    sex="male",
    height_cm=180,
    birth_date=date(2000, 1, 1),
    tdee_override=2200,
    meals_per_day=3,
    activity_level="moderate",
    goal="weight_loss"
)
result = build_user_nutrition_profile(user)
result

{'line_user_id': 'test4',
 'email': None,
 'first_name': 'A',
 'last_name': 'B',
 'nickname': 'C',
 'weight_kg': 95,
 'sex': 'male',
 'height_cm': 180,
 'birth_date': '2000-01-01',
 'body_fat_percent': None,
 'meals_per_day': 3,
 'activity_level': 'moderate',
 'goal': 'weight_loss',
 'concern': None,
 'favorite': None,
 'age': 26,
 'bmr': 1950,
 'bmr_method': 'mifflin_st_jeor',
 'tdee': 2200,
 'tdee_method': 'user_override',
 'goal_calorie_adjustment': -500,
 'daily_calories_intake': 1700,
 'protein_g_per_kg_range': [2.0, 2.4],
 'fat_g_per_kg_range': [0.6, 0.8],
 'protein_g_per_kg_selected': 2.2,
 'fat_g_per_kg_selected': 0.7,
 'protein_g_per_day': 209,
 'carb_g_per_day': 68,
 'fat_g_per_day': 66,
 'fiber_g_per_day': 30,
 'sodium_mg_per_day': 2000,
 'protein_kcal_per_day': 836,
 'carb_kcal_per_day': 272,
 'fat_kcal_per_day': 594,
 'protein_g_per_meal': 70,
 'carb_g_per_meal': 23,
 'fat_g_per_meal': 22,
 'fiber_g_per_meal': 10,
 'sodium_mg_per_meal': 667,
 'calories_per_meal': 567,
 'pr

## ✅ 5. Aggressive cut (edge macro case)

In [ ]:
user = UserProfileInput(
    line_user_id="test5",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=95,
    sex="male",
    height_cm=180,
    birth_date=date(2000, 1, 1),
    meals_per_day=4,
    activity_level="moderate",
    goal="aggressive_cut"
)
result = build_user_nutrition_profile(user)
result

{'line_user_id': 'test5',
 'email': None,
 'first_name': 'A',
 'last_name': 'B',
 'nickname': 'C',
 'weight_kg': 95,
 'sex': 'male',
 'height_cm': 180,
 'birth_date': '2000-01-01',
 'body_fat_percent': None,
 'meals_per_day': 4,
 'activity_level': 'moderate',
 'goal': 'aggressive_cut',
 'concern': None,
 'favorite': None,
 'age': 26,
 'bmr': 1950,
 'bmr_method': 'mifflin_st_jeor',
 'tdee': 3022,
 'tdee_method': 'bmr_x_activity_multiplier',
 'goal_calorie_adjustment': -700,
 'daily_calories_intake': 2322,
 'protein_g_per_kg_range': [2.2, 2.6],
 'fat_g_per_kg_range': [0.5, 0.6],
 'protein_g_per_kg_selected': 2.4000000000000004,
 'fat_g_per_kg_selected': 0.55,
 'protein_g_per_day': 228,
 'carb_g_per_day': 236,
 'fat_g_per_day': 52,
 'fiber_g_per_day': 33,
 'sodium_mg_per_day': 2000,
 'protein_kcal_per_day': 912,
 'carb_kcal_per_day': 944,
 'fat_kcal_per_day': 468,
 'protein_g_per_meal': 57,
 'carb_g_per_meal': 59,
 'fat_g_per_meal': 13,
 'fiber_g_per_meal': 8,
 'sodium_mg_per_meal': 500,


## ✅ 6. Sedentary (low protein)

In [ ]:
user = UserProfileInput(
    line_user_id="test6",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=70,
    sex="female",
    height_cm=160,
    birth_date=date(1995, 1, 1),
    meals_per_day=3,
    activity_level="sedentary",
    goal="health"
)
result = build_user_nutrition_profile(user)
result

{'line_user_id': 'test6',
 'email': None,
 'first_name': 'A',
 'last_name': 'B',
 'nickname': 'C',
 'weight_kg': 70,
 'sex': 'female',
 'height_cm': 160,
 'birth_date': '1995-01-01',
 'body_fat_percent': None,
 'meals_per_day': 3,
 'activity_level': 'sedentary',
 'goal': 'health',
 'concern': None,
 'favorite': None,
 'age': 31,
 'bmr': 1384,
 'bmr_method': 'mifflin_st_jeor',
 'tdee': 1661,
 'tdee_method': 'bmr_x_activity_multiplier',
 'goal_calorie_adjustment': 0,
 'daily_calories_intake': 1661,
 'protein_g_per_kg_range': [0.8, 1.0],
 'fat_g_per_kg_range': [0.8, 1.0],
 'protein_g_per_kg_selected': 0.9,
 'fat_g_per_kg_selected': 0.9,
 'protein_g_per_day': 63,
 'carb_g_per_day': 210,
 'fat_g_per_day': 63,
 'fiber_g_per_day': 25,
 'sodium_mg_per_day': 2000,
 'protein_kcal_per_day': 252,
 'carb_kcal_per_day': 840,
 'fat_kcal_per_day': 567,
 'protein_g_per_meal': 21,
 'carb_g_per_meal': 70,
 'fat_g_per_meal': 21,
 'fiber_g_per_meal': 8,
 'sodium_mg_per_meal': 667,
 'calories_per_meal': 554

## ❌ 7. Invalid input (ควร error)

In [ ]:
user = UserProfileInput(
    line_user_id="test7",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=-10,   # ❌ invalid
    sex="male",
    height_cm=180,
    birth_date=date(2000, 1, 1),
)

result = build_user_nutrition_profile(user)
result

ValueError: weight_kg must be > 0

## ❌ 8. Body fat เกิน range

In [ ]:
user = UserProfileInput(
    line_user_id="test8",
    email=None,
    first_name="A",
    last_name="B",
    nickname="C",
    weight_kg=95,
    sex="male",
    height_cm=180,
    birth_date=date(2000, 1, 1),
    body_fat_percent=90  # ❌
)
result = build_user_nutrition_profile(user)
result

ValueError: body_fat_percent should be between 3 and 70